# H5-OmniFusion: Publication Ready (Deterministic + Parallel)\n\nThis notebook implements the **H5-OmniFusion** architecture with **100% deterministic reproducibility**.\n\n### Publication Features:\n*   **Reproducibility**: Strict seeding (Seed 42) ensures identical results every run.\n*   **Checkpointing**: Saves best models directly to **Google Drive**.\n*   **Visualization**: Generates professional Line Graphs (Loss/F1) and Confusion Matrices.\n\n### Instructions:\n1.  **Mount Drive**: Grant permission when prompted.\n2.  **Set Directories**: Update `DATA_DIR` and `DRIVE_SAVE_DIR` in the config cell.\n3.  **Run All Cells**: Training, Optimization, and Plotting happen automatically.

## Phase 0: Environment & Drive Setup

In [ ]:
import os\nimport random\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.optim as optim\nimport torch.nn.functional as F\nimport matplotlib.pyplot as plt\nimport seaborn as sns\nfrom torch.utils.data import Dataset, DataLoader, Subset\nfrom sklearn.metrics import f1_score, roc_auc_score, mean_absolute_error, confusion_matrix, roc_curve\nfrom sklearn.model_selection import KFold\nfrom google.colab import drive\n\n# Mount Drive\ntry:\n    drive.mount('/content/drive')\n    print("Google Drive mounted successfully.")\nexcept Exception as e:\n    print("Warning: Drive mount failed (are you local?). Checkpoints will save locally.")\n\ndef set_deterministic(seed=42):\n    print(f"[Phase 0] Setting Strict Determinism with Seed {seed}...")\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    torch.cuda.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.deterministic = True\n    torch.backends.cudnn.benchmark = False\n    os.environ['PYTHONHASHSEED'] = str(seed)\n    print("Deterministic environment initialized.")\n\nset_deterministic(42)

## Phase 1: Data Ingestion

In [ ]:
class H5PrecomputedDataset(Dataset):\n    def __init__(self, data_dir):\n        self.data_dir = data_dir\n        print(f"[Phase 1] Loading tensors from {data_dir}...")\n        try:\n            self.audio = torch.load(os.path.join(data_dir, 'audio_tensors.pt'))\n            self.video = torch.load(os.path.join(data_dir, 'video_tensors.pt'))\n            self.text = torch.load(os.path.join(data_dir, 'text_tensors.pt'))\n            self.face = torch.load(os.path.join(data_dir, 'face_tensors.pt'))\n            self.tabular = torch.load(os.path.join(data_dir, 'tabular_tensors.pt'))\n            self.labels = torch.load(os.path.join(data_dir, 'labels.pt')) \n            self.quality = torch.ones((len(self.labels), 5)) # Proxy quality\n            if os.path.exists(os.path.join(data_dir, 'quality_scores.pt')):\n                self.quality = torch.load(os.path.join(data_dir, 'quality_scores.pt'))\n        except FileNotFoundError:\n            print("WARNING: Data files not found. Using MOCK DATA.")\n            N = 200\n            self.audio = torch.randn(N, 50, 768)\n            self.video = torch.randn(N, 50, 768)\n            self.text = torch.randn(N, 50, 768)\n            self.face = torch.randn(N, 50, 768)\n            self.tabular = torch.randn(N, 768)\n            self.labels = torch.stack([torch.randint(0, 2, (N,)).float(), torch.rand(N)*24], dim=1)\n            self.quality = torch.rand(N, 5)\n\n    def __len__(self): return len(self.labels)\n    def __getitem__(self, idx):\n        return {\n            'audio': self.audio[idx], 'video': self.video[idx], 'text': self.text[idx],\n            'face': self.face[idx], 'tabular': self.tabular[idx], 'quality': self.quality[idx],\n            'binary_label': self.labels[idx][0], 'score_label': self.labels[idx][1]\n        }

## Phase 2: H5-OmniFusion Architecture

In [ ]:
class ModalityEncoder(nn.Module):\n    def __init__(self, mode='lstm'):\n        super().__init__()\n        self.mode = mode\n        if mode == 'lstm': self.enc = nn.LSTM(768, 384, num_layers=2, batch_first=True, bidirectional=True)\n        elif mode == 'transformer':\n             self.enc = nn.TransformerEncoder(nn.TransformerEncoderLayer(768, 8, batch_first=True), 1)\n        elif mode == 'mlp': self.enc = nn.Sequential(nn.Linear(768,512), nn.LayerNorm(512), nn.ReLU(), nn.Linear(512,768))\n    def forward(self, x):\n        if self.mode == 'lstm': return self.enc(x)[0][:,-1,:]\n        elif self.mode == 'transformer': return self.enc(x).mean(dim=1)\n        return self.enc(x)\n\nclass HypergraphFusion(nn.Module):\n    def __init__(self):\n        super().__init__()\n        self.W = nn.Linear(768, 768)\n        self.register_buffer('H', torch.tensor([[1,0,1],[1,0,1],[0,1,1],[0,1,1],[0,0,1]], dtype=torch.float32))\n    def forward(self, inputs):\n        X = torch.stack(inputs, dim=1)\n        X_W = self.W(X)\n        Dv, De = self.H.sum(1).view(-1,1,1), self.H.sum(0).view(1,-1,1)\n        X_edge = (self.H.t() @ X_W) / De\n        return F.relu((self.H @ X_edge) / Dv) + X\n\nclass QualityGatedMoE(nn.Module):\n    def __init__(self):\n        super().__init__()\n        self.experts = nn.ModuleList([nn.Sequential(nn.Linear(768,512), nn.ReLU(), nn.Linear(512,768)) for _ in range(5)])\n        self.gate = nn.Linear(768, 5)\n    def forward(self, x, q):\n        glob = x.mean(1)\n        w = F.softmax(self.gate(glob) * q, dim=1)\n        top_k_w, top_k_idx = torch.topk(w, 2, dim=1)\n        top_k_w = top_k_w / top_k_w.sum(1, keepdim=True)\n        out = torch.zeros_like(glob)\n        for i in range(2):\n            idx = top_k_idx[:,i]\n            out += top_k_w[:,i].unsqueeze(1) * torch.stack([self.experts[j](glob) for j in range(5)],1)[torch.arange(len(idx)), idx]\n        return out\n\nclass H5OmniFusion(nn.Module):\n    def __init__(self, checkpoint=None):\n        super().__init__()\n        self.audio, self.video, self.face = [ModalityEncoder('lstm') for _ in range(3)]\n        self.text, self.tab = ModalityEncoder('transformer'), ModalityEncoder('mlp')\n        self.hg, self.moe = HypergraphFusion(), QualityGatedMoE()\n        self.cls, self.reg = nn.Linear(768, 1), nn.Linear(768, 1)\n        if checkpoint and os.path.exists(checkpoint):\n             self.load_state_dict(torch.load(checkpoint, map_location='cpu'), strict=False)\n             print("Checkpoint loaded.")\n    def forward(self, b):\n        emb = [self.audio(b['audio']), self.text(b['text']), self.video(b['video']), self.face(b['face']), self.tab(b['tabular'])]\n        fused = self.hg(emb)\n        final = self.moe(fused, b['quality'])\n        return self.cls(final).squeeze(-1), self.reg(final).squeeze(-1)

## Phase 3: Training with Visualization

In [ ]:
class FocalLoss(nn.Module):\n    def __init__(self, alpha=1, gamma=2):\n        super().__init__()\n        self.alpha, self.gamma = alpha, gamma\n    def forward(self, inp, tar):\n        bce = F.binary_cross_entropy_with_logits(inp, tar, reduction='none')\n        return torch.mean(self.alpha * (1 - torch.exp(-bce))**self.gamma * bce)\n\ndef train_fold(fold, train_idx, val_idx, ds, cfg):\n    train_loader = DataLoader(Subset(ds, train_idx), batch_size=32, shuffle=True, worker_init_fn=lambda x: np.random.seed(42+x))\n    val_loader = DataLoader(Subset(ds, val_idx), batch_size=32)\n    \n    model = H5OmniFusion(cfg['ckpt']).to(cfg['device'])\n    opt = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)\n    fl, mse = FocalLoss(), nn.MSELoss()\n    \n    best_f1 = 0\n    history = {'train_loss': [], 'val_f1': []}\n    print(f"\n--- Fold {fold} ---")\n    \n    for ep in range(cfg['epochs']):\n        model.train()\n        t_loss = 0\n        for b in train_loader:\n            b = {k:v.to(cfg['device']) for k,v in b.items()}\n            opt.zero_grad()\n            l, s = model(b)\n            loss = 0.7*fl(l, b['binary_label']) + 0.3*mse(s, b['score_label'])\n            loss.backward()\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n            opt.step()\n            t_loss += loss.item()\n            \n        # Validation\n        val_res = evaluate(model, val_loader, cfg['device'], plot=False)\n        history['train_loss'].append(t_loss/len(train_loader))\n        history['val_f1'].append(val_res['f1'])\n        \n        if val_res['f1'] > best_f1:\n            best_f1 = val_res['f1']\n            path = os.path.join(cfg['save_dir'], f"h5_fold{fold}_best.pt")\n            torch.save(model.state_dict(), path)\n            \n        print(f"Ep {ep+1} | Loss: {history['train_loss'][-1]:.4f} | Val F1: {val_res['f1']:.4f}")\n        \n    plot_history(history, fold)\n    return model\n\ndef evaluate(model, loader, device, plot=True):\n    model.eval()\n    probs, labs = [], []\n    with torch.no_grad():\n        for b in loader:\n            b = {k:v.to(device) for k,v in b.items()}\n            probs.extend(torch.sigmoid(model(b)[0]).cpu().numpy())\n            labs.extend(b['binary_label'].cpu().numpy())\n            \n    fpr, tpr, thr = roc_curve(labs, probs)\n    best_thr = thr[np.argmax(tpr-fpr)]\n    preds = (np.array(probs) >= best_thr).astype(int)\n    \n    if plot:\n        plot_confusion_matrix(labs, preds)\n        print(f"Best Threshold: {best_thr:.4f}")\n        \n    return {'f1': f1_score(labs, preds), 'auc': roc_auc_score(labs, probs)}\n\n# --- VISUALIZATION HELPERS ---\ndef plot_history(hist, fold):\n    fig, ax = plt.subplots(1, 2, figsize=(12, 4))\n    ax[0].plot(hist['train_loss'], label='Train Loss', color='b')\n    ax[0].set_title(f'Fold {fold} Training Loss')\n    ax[1].plot(hist['val_f1'], label='Val F1', color='g')\n    ax[1].set_title(f'Fold {fold} Validation F1')\n    plt.show()\n\ndef plot_confusion_matrix(y_true, y_pred):\n    cm = confusion_matrix(y_true, y_pred)\n    plt.figure(figsize=(6,5))\n    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['No Dep', 'Dep'], yticklabels=['No Dep', 'Dep'])\n    plt.ylabel('True Label')\n    plt.xlabel('Predicted Label')\n    plt.title('Confusion Matrix')\n    plt.show()

In [ ]:
# --- MAIN EXECUTION ---\n# Configuration\nCONFIG = {\n    'data_dir': './data', # <--- LOCAL PATH\n    'save_dir': '/content/drive/MyDrive/H5_Checkpoints', # <--- DRIVE PATH\n    'ckpt': None, # Path to pre-trained weights if any\n    'epochs': 30,\n    'device': 'cuda' if torch.cuda.is_available() else 'cpu'\n}\n\nos.makedirs(CONFIG['save_dir'], exist_ok=True)\n\n# Run\nds = H5PrecomputedDataset(CONFIG['data_dir'])\nkf = KFold(5, shuffle=True, random_state=42)\n\nfor fold, (t_idx, v_idx) in enumerate(kf.split(ds)):\n    # To run only specific fold, add: if fold != 0: continue\n    print(f"Running Fold {fold}...")\n    model = train_fold(fold, t_idx, v_idx, ds, CONFIG)\n    # Final Evaluation Plot\n    print("Final Evaluation:")\n    evaluate(model, DataLoader(Subset(ds, v_idx), batch_size=32), CONFIG['device'], plot=True)